In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
os.chdir('/content/drive/MyDrive/audio_dataset')

In [2]:
import librosa
import tensorflow as tf
import numpy as np
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from keras.utils import to_categorical

In [3]:

# Load audio dataset
train_set, val_set = tf.keras.utils.audio_dataset_from_directory(
    directory="/content/drive/MyDrive/audio_dataset",
    batch_size=16,
    validation_split=0.2,
    output_sequence_length=16000,
    seed=0,
    subset='both',
    shuffle=True
)

label_names = np.array(train_set.class_names)
print("Labels:", label_names)
print("Number of classes:", len(label_names))

Found 7469 files belonging to 4 classes.
Using 5976 files for training.
Using 1493 files for validation.
Labels: ['backward' 'house' 'marvin' 'visual']
Number of classes: 4


In [ ]:
# Load audio dataset
train_set, val_set = tf.keras.utils.audio_dataset_from_directory(
    directory="/content/drive/MyDrive/audio_dataset",
    batch_size=16,
    validation_split=0.2,
    output_sequence_length=16000,
    seed=0,
    subset='both',
    shuffle=True
)

label_names = np.array(train_set.class_names)
print("Labels:", label_names)
print("Number of classes:", len(label_names))

# Squeeze audio
def squeeze(audio, labels):
    audio = tf.squeeze(audio, axis=-1)
    return audio, labels

# train_set = train_set.map(squeeze, tf.data.AUTOTUNE)
# val_set = val_set.map(squeeze, tf.data.AUTOTUNE)


# Extract mel-spectrogram
def extract_mel(audio):
    mel = librosa.feature.melspectrogram(y=audio, sr=16000, n_mels=110)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db

# Convert entire dataset to arrays
print("\nConverting audio to mel-spectrograms...")

X_train = []
y_train = []
for audio_batch, label_batch in train_set:
    for audio, label in zip(audio_batch.numpy(), label_batch.numpy()):
        mel = extract_mel(audio)
        # Resize to 110x110
        mel_resized = tf.image.resize(mel[..., np.newaxis], (110, 110)).numpy()
        # Convert to 3 channels
        mel_3ch = np.repeat(mel_resized, 3, axis=-1)
        X_train.append(mel_3ch)
        y_train.append(label)

X_test = []
y_test = []
for audio_batch, label_batch in val_set:
    for audio, label in zip(audio_batch.numpy(), label_batch.numpy()):
        mel = extract_mel(audio)
        mel_resized = tf.image.resize(mel[..., np.newaxis], (110, 110)).numpy()
        mel_3ch = np.repeat(mel_resized, 3, axis=-1)
        X_test.append(mel_3ch)
        y_test.append(label)

# Convert to numpy arrays
X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_test)}")

# Normalize data
X_train = (X_train - X_train.mean()) / (X_train.std() + 1e-6)
X_test = (X_test - X_test.mean()) / (X_test.std() + 1e-6)

# One-hot encode labels
num_classes = len(label_names)
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

# Build CNN Model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(110, 110, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.3),

    Conv2D(256, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.3),

    Flatten(),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(num_classes, activation='`')
])

# Compile model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nTraining model...")

# Train model
history = model.fit(
    X_train, y_train_cat,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test_cat),
    verbose=1
)

# Evaluate
loss, acc = model.evaluate(X_test, y_test_cat)
print(f"\n✅ Final Validation Accuracy: {acc * 100:.2f}%")

# Save model
model.save('audio_word_classifier.keras')
print("\n✅ Model saved as 'audio_word_classifier.keras'")

Found 7469 files belonging to 4 classes.
Using 5976 files for training.
Using 1493 files for validation.
Labels: ['backward' 'house' 'marvin' 'visual']
Number of classes: 4

Converting audio to mel-spectrograms...
Training samples: 5976
Validation samples: 1493


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Training model...
Epoch 1/10
187/187 ━━━━━━━━━━━━━━━━━━━━ 281s 1s/step - accuracy: 0.7118 - loss: 0.9589 - val_accuracy: 0.3523 - val_loss: 3.9780
Epoch 2/10
187/187 ━━━━━━━━━━━━━━━━━━━━ 272s 1s/step - accuracy: 0.8774 - loss: 0.3593 - val_accuracy: 0.7783 - val_loss: 0.6723
Epoch 3/10
187/187 ━━━━━━━━━━━━━━━━━━━━ 335s 2s/step - accuracy: 0.9053 - loss: 0.2622 - val_accuracy: 0.8473 - val_loss: 0.4667
Epoch 4/10
187/187 ━━━━━━━━━━━━━━━━━━━━ 314s 1s/step - accuracy: 0.9165 - loss: 0.2420 - val_accuracy: 0.8473 - val_loss: 0.5438
Epoch 5/10
187/187 ━━━━━━━━━━━━━━━━━━━━ 266s 1s/step - accuracy: 0.9333 - loss: 0.1852 - val_accuracy: 0.9585 - val_loss: 0.1143
Epoch 6/10
187/187 ━━━━━━━━━━━━━━━━━━━━ 274s 1s/step - accuracy: 0.9472 - loss: 0.1709 - val_accuracy: 0.9632 - val_loss: 0.1132
Epoch 7/10
187/187 ━━━━━━━━━━━━━━━━━━━━ 262s 1s/step - accuracy: 0.9543 - loss: 0.1350 - val_accuracy: 0.9625 - val_loss: 0.1148
Epoch 8/10
187/187 ━━━━━━━━━━━━━━━━━━━━ 326s 1s/step - accuracy: 0.9606 - loss

In [4]:
# Squeeze audio
def squeeze(audio, labels):
    audio = tf.squeeze(audio, axis=-1)
    return audio, labels

# train_set = train_set.map(squeeze, tf.data.AUTOTUNE)
# val_set = val_set.map(squeeze, tf.data.AUTOTUNE)


# Extract mel-spectrogram
def extract_mel(audio):
    mel = librosa.feature.melspectrogram(y=audio, sr=16000, n_mels=110)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db

In [ ]:
# Save model
model.save('/content/drive/MyDrive/audio_word_classifier.keras')

In [6]:
# =====================================================
# Inference / Testing (NO CHANGE TO MODEL)
# =====================================================


model = tf.keras.models.load_model('/content/drive/MyDrive/audio_word_classifier.keras')


def preprocess_audio_for_test(audio_path):
    """
    Preprocess a single audio file exactly like training data.
    """
    audio, sr = librosa.load(audio_path, sr=16000)

    # Pad or trim to 1 second
    if len(audio) < 16000:
        audio = np.pad(audio, (0, 16000 - len(audio)))
    else:
        audio = audio[:16000]

    # Extract mel-spectrogram (same as training)
    mel = extract_mel(audio)

    # Resize to 110x110
    mel_resized = tf.image.resize(mel[..., np.newaxis], (110, 110)).numpy()

    # Convert to 3 channels (same as training)
    mel_3ch = np.repeat(mel_resized, 3, axis=-1)

    # Normalize using same logic as training
    mel_3ch = (mel_3ch - mel_3ch.mean()) / (mel_3ch.std() + 1e-6)

    # Add batch dimension
    return np.expand_dims(mel_3ch, axis=0)


def classify_audio(audio_path):
    """
    Classify a single audio file and return label and confidence.
    """
    x = preprocess_audio_for_test(audio_path)
    predictions = model.predict(x, verbose=0)[0]

    predicted_index = np.argmax(predictions)
    predicted_label = label_names[predicted_index]
    confidence = predictions[predicted_index]
    return predicted_label, confidence, predictions


# ============================
# Example Usage
# ============================
test_audio_path = "visual.ogg" # Corrected path
label, confidence, probs = classify_audio(test_audio_path)
print("Predicted Label:", label)
print(f"Confidence: {confidence * 100:.2f}%")
print("All Probabilities:", probs)

Predicted Label: visual
Confidence: 100.00%
All Probabilities: [4.4506228e-06 3.5876377e-09 9.0812745e-07 9.9999464e-01]
